# Stretch-corrected video synchronization

This notebook is the publishable drift-correction companion to `video_aligner_publish_2.py`. It is intended for the case where two videos look aligned in the middle, but the child camera is ahead of the parent near the beginning and lags behind near the end because one stream is slightly stretched or compressed.

Workflow: first spectrogram alignment and cut; residual front/middle/back alignment on the first-cut clips; second cut to the shared overlap from the original source videos; audio-only tempo correction; optional verification on the corrected outputs.


In [1]:
from __future__ import annotations

import json
import logging
import math
import os
import pickle
import shutil
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from video_aligner_publish_2 import (
    DEFAULT_N_MELS,
    DEFAULT_SPECTROGRAM_SR,
    FrameAligner,
    _discover_video_stems,
    _resolve_torch_device,
    setup_logging,
)


## Configuration

The file naming convention matches the README: videos and WAVs are named `{subject}_{camera}.mp4` and `{subject}_{camera}.wav`. Leave `subject_ids` or `camera_ids` as `None` to discover them from the input folders or from previous first-cut outputs.


In [ ]:
@dataclass
class SyncConfig:
    input_video_dir: Path
    input_audio_dir: Path
    output_root: Path
    input_world_timestamp_dir: Path | None = None
    subject_ids: list[str] | None = None
    camera_ids: list[str] | None = None
    first_ref_cam: str | None = None
    drift_ref_cam: str | None = "parent"
    corrected_cameras: list[str] | None = None
    run_first_alignment: bool = True
    device: str = "auto"
    max_lag_sec: float = 2.0
    audio_sr: int | None = None
    spectrogram_sr: float = DEFAULT_SPECTROGRAM_SR
    n_mels: int = DEFAULT_N_MELS
    residual_window_sec: float = 10.0
    residual_search_sec: float = 2.0
    min_drift_to_correct_sec: float = 0.02
    max_speed_delta: float = 0.05
    video_crf: int = 18
    video_preset: str = "medium"
    audio_bitrate: str = "192k"

os.environ["GBAT_VIDEO_DIR"] = "Your/Path/To/Video/Directory"
os.environ["GBAT_AUDIO_DIR"] = "Your/Path/To/Video/Directory"
os.environ["GBAT_SYNC_OUTPUT_DIR"] = "Your/Path/To/Video/Directory"


PROJECT_DIR = Path.cwd()
CONFIG = SyncConfig(
    input_video_dir=Path(os.environ.get("GBAT_VIDEO_DIR", PROJECT_DIR / "input_videos")),
    input_audio_dir=Path(os.environ.get("GBAT_AUDIO_DIR", PROJECT_DIR / "input_audios")),
    input_world_timestamp_dir=None,
    output_root=Path(os.environ.get("GBAT_SYNC_OUTPUT_DIR", PROJECT_DIR / "stretch_corrected_output")),
    subject_ids=None,
    camera_ids=None,
    first_ref_cam=None,
    drift_ref_cam="parent",
    corrected_cameras=None,
    audio_sr = 48000
)


## Shared Helpers

The residual offsets below use the same sign convention as `FrameAligner.compute_time_shift`: the offset is `reference_time - comparison_time`. A positive value means the comparison camera's matching audio occurs earlier than the reference; a negative value means it occurs later. With `parent` as reference and `child` as comparison, the pattern "child ahead at the beginning and lagging at the end" should appear as positive `front_offset_sec`, near-zero `middle_offset_sec`, and negative `back_offset_sec`.


In [3]:
def _id_text(value) -> str:
    return str(value).strip()


def stage1_cut_dir(config: SyncConfig) -> Path:
    return config.output_root / "01_first_alignment"


def stage1_merged_dir(config: SyncConfig) -> Path:
    return config.output_root / "01_first_alignment_merged"


def stage2_dir(config: SyncConfig) -> Path:
    return config.output_root / "02_second_cut_stretch_corrected"


def diagnostics_dir(config: SyncConfig) -> Path:
    return config.output_root / "diagnostics"


def first_cut_audio_path(config: SyncConfig, subject: str, camera: str) -> Path:
    return stage1_cut_dir(config) / f"{subject}_{camera}_cut.wav"


def first_cut_merged_path(config: SyncConfig, subject: str, camera: str) -> Path:
    return stage1_merged_dir(config) / f"{subject}_{camera}_cut_merged.mp4"


def corrected_output_path(config: SyncConfig, subject: str, camera: str) -> Path:
    return stage2_dir(config) / f"{subject}_{camera}_cut2_corrected.mp4"


def run_command(cmd: list[str | Path | float | int], context: str) -> subprocess.CompletedProcess:
    cmd = [str(part) for part in cmd]
    try:
        return subprocess.run(cmd, check=True, text=True, capture_output=True)
    except subprocess.CalledProcessError as exc:
        print(f"Command failed during {context}:")
        print(" ".join(cmd))
        if exc.stderr:
            print(exc.stderr[-4000:])
        raise


def require_tool(name: str) -> None:
    if shutil.which(name) is None:
        raise RuntimeError(f"Required command-line tool is missing: {name}")


def discover_cut_audio_stems(cut_dir: Path) -> list[tuple[str, str]]:
    pairs: list[tuple[str, str]] = []
    if not cut_dir.exists():
        return pairs
    for wav_path in sorted(cut_dir.glob("*_cut.wav")):
        stem = wav_path.stem
        if not stem.endswith("_cut"):
            continue
        raw_stem = stem[:-4]
        parts = raw_stem.split("_", 1)
        if len(parts) == 2 and parts[0] and parts[1]:
            pairs.append((_id_text(parts[0]), _id_text(parts[1])))
    return pairs


def resolve_subjects_and_cameras(config: SyncConfig) -> tuple[list[str], list[str]]:
    pairs: list[tuple[str, str]] = []
    if config.input_video_dir.exists():
        pairs.extend(_discover_video_stems(config.input_video_dir))
    pairs.extend(discover_cut_audio_stems(stage1_cut_dir(config)))

    if config.subject_ids is None:
        subjects = sorted({subject for subject, _ in pairs})
    else:
        subjects = [_id_text(subject) for subject in config.subject_ids]

    if config.camera_ids is None:
        cameras = sorted({camera for _, camera in pairs})
    else:
        cameras = [_id_text(camera) for camera in config.camera_ids]

    if not subjects:
        raise ValueError("No subjects found. Set CONFIG.subject_ids or check CONFIG.input_video_dir / first-cut outputs.")
    if not cameras:
        raise ValueError("No cameras found. Set CONFIG.camera_ids or check CONFIG.input_video_dir / first-cut outputs.")
    return subjects, cameras


def media_duration_sec(path: Path) -> float:
    payload = run_command(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration",
            "-of",
            "json",
            path,
        ],
        context="probe media duration",
    ).stdout
    duration = json.loads(payload).get("format", {}).get("duration")
    if duration in (None, "N/A"):
        raise ValueError(f"Could not read duration from {path}")
    return float(duration)


def choose_drift_reference(config: SyncConfig, subject: str, cameras: list[str]) -> str | None:
    available = [camera for camera in cameras if first_cut_audio_path(config, subject, camera).exists()]
    if not available:
        return None
    if config.drift_ref_cam in available:
        return _id_text(config.drift_ref_cam)
    if config.first_ref_cam in available:
        return _id_text(config.first_ref_cam)
    return available[0]


def corrected_camera_set(config: SyncConfig, cameras: list[str], drift_ref_cam: str) -> set[str]:
    if config.corrected_cameras is None:
        return {camera for camera in cameras if camera != drift_ref_cam}
    return {_id_text(camera) for camera in config.corrected_cameras if _id_text(camera) != drift_ref_cam}


def fmt_float(value: float) -> str:
    text = f"{float(value):.9f}".rstrip("0").rstrip(".")
    return text if text else "0"


require_tool("ffmpeg")
require_tool("ffprobe")
CONFIG.output_root.mkdir(parents=True, exist_ok=True)
SUBJECTS, CAMERAS = resolve_subjects_and_cameras(CONFIG)
print(f"Subjects: {SUBJECTS}")
print(f"Cameras: {CAMERAS}")


Subjects: ['expand0p1pctoffset1s', 'expand0p1pctoffset50ms', 'expand1pctoffset1s', 'expand1pctoffset50ms']
Cameras: ['child', 'parent']


## First Alignment And Cut

This stage delegates to `FrameAligner` from `video_aligner_publish_2.py`: load WAV files, estimate spectrogram offsets, convert cut times to frame indices using world timestamps or ffprobe, cut video/audio, merge, and save `audio_offset.pkl` plus `to_cut_frames.pkl`.


In [4]:
def run_first_alignment_and_cut(config: SyncConfig, subjects: list[str], cameras: list[str]) -> FrameAligner | None:
    if not config.run_first_alignment:
        print("Skipping first alignment because CONFIG.run_first_alignment is False.")
        return None

    if not config.input_video_dir.exists():
        raise FileNotFoundError(f"Input video directory does not exist: {config.input_video_dir}")
    if not config.input_audio_dir.exists():
        raise FileNotFoundError(f"Input audio directory does not exist: {config.input_audio_dir}")

    output_dir = stage1_cut_dir(config)
    merged_output_dir = stage1_merged_dir(config)
    output_dir.mkdir(parents=True, exist_ok=True)
    merged_output_dir.mkdir(parents=True, exist_ok=True)

    log_path = output_dir / "video_aligner_first_alignment.log"
    setup_logging(log_path)
    logger = logging.getLogger("first_alignment")
    device, requested_device = _resolve_torch_device(config.device, logger)

    aligner = FrameAligner(
        config.input_video_dir,
        config.input_audio_dir,
        output_dir,
        merged_output_dir,
        input_world_timestamp_dir=config.input_world_timestamp_dir,
        logger=logger,
        device=device,
        requested_device=requested_device,
        max_lag_sec=config.max_lag_sec,
        audio_sr=config.audio_sr,
        spectrogram_sr=config.spectrogram_sr,
        n_mels=config.n_mels,
    )

    for subject in subjects:
        logger.info("Starting first alignment for subject=%s", subject)
        aligner.process_subject(subject, cameras, config.first_ref_cam)
        logger.info("Completed first alignment for subject=%s", subject)

    aligner.save_audio_offsets_pickle(output_dir / "audio_offset.pkl")
    aligner.save_cut_frames_pickle(output_dir / "to_cut_frames.pkl")
    return aligner


first_aligner = run_first_alignment_and_cut(CONFIG, SUBJECTS, CAMERAS)


2026-06-17 20:40:10,842 | INFO | first_alignment | Starting first alignment for subject=expand0p1pctoffset1s



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/summer_2026/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/summer_2026/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/summer_2026/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/summer_2026/lib/python3.10/site-packages/traitlets/config/application.py", line 1082, in launc

2026-06-17 20:40:20,282 | INFO | first_alignment | Loaded audio=expand0p1pctoffset1s_child.wav sr_orig=48000 sr_target=48000 sr_used=48000 len_resampled=(5814649,) len_original=(5814649,)
2026-06-17 20:40:28,392 | INFO | first_alignment | Loaded audio=expand0p1pctoffset1s_parent.wav sr_orig=48000 sr_target=48000 sr_used=48000 len_resampled=(5760000,) len_original=(5760000,)
2026-06-17 20:40:28,401 | INFO | first_alignment | Loaded waveforms subject=expand0p1pctoffset1s cameras=['child', 'parent'] elapsed_sec=17.558
2026-06-17 20:40:28,402 | INFO | first_alignment | ------------------------------------
2026-06-17 20:40:28,402 | INFO | first_alignment | Start Spectrogram Synchronization
2026-06-17 20:40:28,403 | INFO | first_alignment | ------------------------------------
2026-06-17 20:40:28,403 | INFO | first_alignment | subject=expand0p1pctoffset1s ref_cam=child audio_sr=48000 spectrogram_sr=400.000 n_mels=64 spectrogram_hop=120 max_lag_sec=2.000 pad_size=800
2026-06-17 20:40:30,104 |

## Second Alignment: Measure Residual Drift

After the first cut, the global spectrogram alignment can make the middle look synchronized while the endpoints diverge. This stage measures residual offsets in short front, middle, and back windows of the first-cut audio. The drift correction uses `back_offset_sec - front_offset_sec`; the middle offset is retained as a diagnostic for the original problem.


In [5]:
def make_analysis_aligner(config: SyncConfig) -> FrameAligner:
    diag_dir = diagnostics_dir(config)
    diag_dir.mkdir(parents=True, exist_ok=True)
    log_path = diag_dir / "residual_alignment.log"
    setup_logging(log_path)
    logger = logging.getLogger("residual_alignment")
    device, requested_device = _resolve_torch_device(config.device, logger)
    return FrameAligner(
        input_video_dir=config.input_video_dir,
        input_audio_dir=stage1_cut_dir(config),
        output_dir=diag_dir,
        merged_output_dir=diag_dir,
        input_world_timestamp_dir=None,
        logger=logger,
        device=device,
        requested_device=requested_device,
        max_lag_sec=config.residual_search_sec,
        audio_sr=config.audio_sr,
        spectrogram_sr=config.spectrogram_sr,
        n_mels=config.n_mels,
    )


def load_audio_mono(path: Path, target_sr: int | None) -> tuple[np.ndarray, int]:
    if not path.exists():
        raise FileNotFoundError(path)
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return audio.astype(np.float32, copy=False), int(sr)


def audio_window(audio: np.ndarray, sr: int, start_sec: float, window_sec: float) -> np.ndarray:
    start = max(0, int(round(start_sec * sr)))
    length = int(round(window_sec * sr))
    stop = min(len(audio), start + length)
    if stop <= start:
        raise ValueError("Requested audio window is empty.")
    window = audio[start:stop]
    if len(window) < max(1, length // 2):
        raise ValueError("Requested audio window is too short for reliable residual alignment.")
    return window


def estimate_offset_sec(
    aligner: FrameAligner,
    ref_audio: np.ndarray,
    cmp_audio: np.ndarray,
    sr: int,
    config: SyncConfig,
) -> tuple[float, np.ndarray]:
    hop_length = max(1, int(round(sr / config.spectrogram_sr)))
    ref_spec = aligner.compute_log_mel_spectrogram(
        ref_audio,
        sr,
        n_mels=config.n_mels,
        hop_length=hop_length,
    ).astype(np.float32, copy=False)
    cmp_spec = aligner.compute_log_mel_spectrogram(
        cmp_audio,
        sr,
        n_mels=config.n_mels,
        hop_length=hop_length,
    ).astype(np.float32, copy=False)

    ref_spec = ref_spec - ref_spec.mean(axis=-1, keepdims=True)
    cmp_spec = cmp_spec - cmp_spec.mean(axis=-1, keepdims=True)
    lag_frames, corr = aligner.compute_time_shift(
        ref_spec,
        cmp_spec,
        max_lag_sec=config.residual_search_sec,
        spectrograph_sr=config.spectrogram_sr,
    )
    return float(lag_frames) / float(config.spectrogram_sr), np.asarray(corr).reshape(-1)


def measure_residual_offsets(config: SyncConfig, subjects: list[str], cameras: list[str]) -> pd.DataFrame:
    aligner = make_analysis_aligner(config)
    rows: list[dict[str, object]] = []

    for subject in subjects:
        drift_ref = choose_drift_reference(config, subject, cameras)
        if drift_ref is None:
            print(f"[skip] subject={subject}: no first-cut audio files found")
            continue

        ref_path = first_cut_audio_path(config, subject, drift_ref)
        ref_audio, sr_ref = load_audio_mono(ref_path, config.audio_sr)
        ref_duration = len(ref_audio) / sr_ref
        cmp_cameras = [camera for camera in cameras if camera != drift_ref]

        for camera in cmp_cameras:
            cmp_path = first_cut_audio_path(config, subject, camera)
            if not cmp_path.exists():
                print(f"[skip] subject={subject} camera={camera}: missing {cmp_path.name}")
                continue
            cmp_audio, sr_cmp = load_audio_mono(cmp_path, config.audio_sr)
            if sr_cmp != sr_ref:
                raise ValueError(f"Sample-rate mismatch for subject={subject} camera={camera}: {sr_ref} vs {sr_cmp}")

            cmp_duration = len(cmp_audio) / sr_cmp
            common_duration = min(ref_duration, cmp_duration)
            if common_duration < config.residual_window_sec * 3:
                print(f"[skip] subject={subject} camera={camera}: clip too short for front/middle/back windows")
                continue

            front_start = 0.0
            middle_start = max(0.0, (common_duration - config.residual_window_sec) / 2.0)
            back_start = max(0.0, common_duration - config.residual_window_sec)
            front_ref = audio_window(ref_audio, sr_ref, front_start, config.residual_window_sec)
            front_cmp = audio_window(cmp_audio, sr_cmp, front_start, config.residual_window_sec)
            middle_ref = audio_window(ref_audio, sr_ref, middle_start, config.residual_window_sec)
            middle_cmp = audio_window(cmp_audio, sr_cmp, middle_start, config.residual_window_sec)
            back_ref = audio_window(ref_audio, sr_ref, back_start, config.residual_window_sec)
            back_cmp = audio_window(cmp_audio, sr_cmp, back_start, config.residual_window_sec)

            front_offset, _ = estimate_offset_sec(aligner, front_ref, front_cmp, sr_ref, config)
            middle_offset, _ = estimate_offset_sec(aligner, middle_ref, middle_cmp, sr_ref, config)
            back_offset, _ = estimate_offset_sec(aligner, back_ref, back_cmp, sr_ref, config)
            front_center = front_start + config.residual_window_sec / 2.0
            middle_center = middle_start + config.residual_window_sec / 2.0
            back_center = back_start + config.residual_window_sec / 2.0
            drift = back_offset - front_offset

            rows.append(
                {
                    "subject": subject,
                    "drift_ref_cam": drift_ref,
                    "camera": camera,
                    "front_offset_sec": front_offset,
                    "middle_offset_sec": middle_offset,
                    "back_offset_sec": back_offset,
                    "drift_sec": drift,
                    "front_center_sec": front_center,
                    "middle_center_sec": middle_center,
                    "back_center_sec": back_center,
                    "analysis_span_sec": back_center - front_center,
                    "ref_duration_sec": ref_duration,
                    "camera_duration_sec": cmp_duration,
                }
            )

    df = pd.DataFrame(rows)
    diagnostics_dir(config).mkdir(parents=True, exist_ok=True)
    df.to_csv(diagnostics_dir(config) / "residual_front_back_offsets.csv", index=False)
    return df


residual_offsets = measure_residual_offsets(CONFIG, SUBJECTS, CAMERAS)
display(residual_offsets)


,subject,drift_ref_cam,camera,front_offset_sec,middle_offset_sec,back_offset_sec,drift_sec,front_center_sec,middle_center_sec,back_center_sec,analysis_span_sec,ref_duration_sec,camera_duration_sec
0,expand0p1pctoffset1s,parent,child,0.0675,0.0150,-0.0425,-0.1100,5.0,60.0,115.0,110.0,120.0,120.065
1,expand0p1pctoffset50ms,parent,child,0.0500,-0.0025,-0.0600,-0.1100,5.0,60.0,115.0,110.0,120.0,120.033
2,expand1pctoffset1s,parent,child,0.8425,-0.3850,-0.3000,-1.1425,5.0,60.0,115.0,110.0,120.0,120.011
3,expand1pctoffset50ms,parent,child,0.8150,-0.3950,-0.3250,-1.1400,5.0,60.0,115.0,110.0,120.0,120.020


In [6]:
def plot_residual_drift(residual_offsets: pd.DataFrame, config: SyncConfig) -> None:
    if residual_offsets.empty:
        print("No residual offsets to plot.")
        return

    plot_dir = diagnostics_dir(config)
    plot_dir.mkdir(parents=True, exist_ok=True)
    for _, row in residual_offsets.iterrows():
        x = [row["front_center_sec"], row["middle_center_sec"], row["back_center_sec"]]
        y = [row["front_offset_sec"], row["middle_offset_sec"], row["back_offset_sec"]]
        plt.figure(figsize=(5, 3), dpi=160)
        plt.plot(x, y, marker="o")
        plt.axhline(0, color="black", linewidth=0.8)
        plt.xlabel("Clip time (sec)")
        plt.ylabel("Residual offset: ref - camera (sec)")
        plt.title(f"Subject {row['subject']} {row['camera']} vs {row['drift_ref_cam']}")
        plt.tight_layout()
        out_path = plot_dir / f"residual_drift_subj{row['subject']}_cam{row['camera']}_ref{row['drift_ref_cam']}.png"
        plt.savefig(out_path)
        plt.close()


plot_residual_drift(residual_offsets, CONFIG)


## Second Cut And Audio Tempo Plan

The second cut plan keeps endpoint alignment, frame extraction, and audio tempo correction separate:

1. Use the measured front and end offsets from the first-cut audio.
2. Compute new start and end cut points in seconds in each original video by adjusting the first-alignment cut points with those offsets.
3. Convert those absolute original-video time points to original frame indices.
4. Cut video from the original file with the same frame-selection method used by `FrameAligner.cut_frames`
5. Cut audio from the original file with the same new start/end time points, then apply `atempo=(span - drift) / span`.
6. Merge the tempo-corrected audio with the frame-cut video.

With the offset convention `reference_time - comparison_time`, each camera's aligned content coordinate is `camera_first_cut_time + offset`. Therefore the new start uses `common_start = max(front_offset)` and `camera_start = common_start - front_offset`. The new end uses `common_end = min(first_cut_duration + back_offset)` and `camera_end = common_end - back_offset`. These first-cut-relative times are added back to the first-alignment original-video start time before converting to original frame indices.


In [7]:
def speed_factor_from_drift(row: pd.Series, config: SyncConfig) -> float:
    span = float(row["analysis_span_sec"])
    if span <= 0:
        raise ValueError(f"Invalid analysis span for row: {row.to_dict()}")
    drift = float(row["drift_sec"])
    if abs(drift) < config.min_drift_to_correct_sec:
        return 1.0
    speed = (span - drift) / span
    if speed <= 0:
        raise ValueError(f"Non-positive speed factor for row: {row.to_dict()}")
    if abs(speed - 1.0) > config.max_speed_delta:
        raise ValueError(
            f"Speed factor {speed:.6f} for subject={row['subject']} camera={row['camera']} is outside "
            f"the configured limit CONFIG.max_speed_delta={config.max_speed_delta}. Inspect residual offsets before running audio tempo correction."
        )
    return float(speed)


def second_cut_video_path(config: SyncConfig, subject: str, camera: str) -> Path:
    return stage2_dir(config) / f"{subject}_{camera}_cut2_video.mp4"


def second_cut_audio_path(config: SyncConfig, subject: str, camera: str) -> Path:
    return stage2_dir(config) / f"{subject}_{camera}_cut2_audio.wav"


def first_cut_frame_map_path(config: SyncConfig) -> Path:
    return stage1_cut_dir(config) / "to_cut_frames.pkl"


def load_first_cut_frame_map(config: SyncConfig) -> dict:
    path = first_cut_frame_map_path(config)
    if not path.exists():
        raise FileNotFoundError(f"First-cut frame map is missing: {path}")
    with open(path, "rb") as f:
        return pickle.load(f)


def first_cut_frame_entry(frame_map: dict, subject: str, camera: str) -> tuple[int, int, float, float]:
    subject_map = frame_map.get(str(subject))
    if subject_map is None:
        raise KeyError(f"No first-cut frame map for subject={subject}")
    entry = subject_map.get(str(camera))
    if entry is None:
        raise KeyError(f"No first-cut frame map for subject={subject} camera={camera}")
    if len(entry) < 4:
        raise ValueError(f"Invalid first-cut frame entry for subject={subject} camera={camera}: {entry}")
    return int(entry[0]), int(entry[1]), float(entry[2]), float(entry[3])


def make_second_pass_aligner(config: SyncConfig) -> FrameAligner:
    stage2_dir(config).mkdir(parents=True, exist_ok=True)
    diagnostics_dir(config).mkdir(parents=True, exist_ok=True)
    log_path = diagnostics_dir(config) / "second_cut_processing.log"
    setup_logging(log_path)
    logger = logging.getLogger("second_cut_processing")
    device, requested_device = _resolve_torch_device(config.device, logger)
    return FrameAligner(
        input_video_dir=config.input_video_dir,
        input_audio_dir=config.input_audio_dir,
        output_dir=stage2_dir(config),
        merged_output_dir=stage2_dir(config),
        input_world_timestamp_dir=config.input_world_timestamp_dir,
        logger=logger,
        device=device,
        requested_device=requested_device,
        max_lag_sec=config.max_lag_sec,
        audio_sr=config.audio_sr,
        spectrogram_sr=config.spectrogram_sr,
        n_mels=config.n_mels,
    )


def original_video_duration_sec(aligner: FrameAligner, config: SyncConfig, subject: str, camera: str) -> float:
    if config.input_world_timestamp_dir is not None:
        timestamp_path = config.input_world_timestamp_dir / f"{subject}_{camera}_world_timestamps.csv"
        if timestamp_path.is_file():
            timestamps = pd.read_csv(timestamp_path)
            timestamp_sec = timestamps["timestamp [ns]"] / 1e9
            return float(timestamp_sec.iloc[-1] - timestamp_sec.iloc[0])

    video_path = aligner._find_video_path(subject, camera)
    if video_path is None:
        raise FileNotFoundError(f"No matching original video found for subject={subject} camera={camera}")
    fps, duration_sec, total_frames = aligner._probe_video_metadata(video_path)
    if duration_sec is None and fps is not None and total_frames is not None:
        duration_sec = total_frames / fps
    if duration_sec is None:
        duration_sec = media_duration_sec(video_path)
    return float(duration_sec)


def frame_bounds_from_original_times(
    aligner: FrameAligner,
    config: SyncConfig,
    subject: str,
    camera: str,
    start_sec: float,
    end_sec: float,
) -> tuple[int, int, float, float]:
    start_sec = float(start_sec)
    end_sec = float(end_sec)
    if end_sec <= start_sec:
        raise ValueError(f"End time must be after start time for subject={subject} camera={camera}: {start_sec} >= {end_sec}")

    source_duration = original_video_duration_sec(aligner, config, subject, camera)
    if end_sec > source_duration + 1e-6:
        raise ValueError(
            f"Second-cut end time exceeds original duration for subject={subject} camera={camera}: "
            f"end={end_sec:.6f}s duration={source_duration:.6f}s"
        )
    cut_end_sec = max(0.0, source_duration - min(end_sec, source_duration))
    start_frame, end_frame, start_rel_time, end_rel_time = aligner.time_to_frame(
        subject,
        camera,
        start_sec,
        cut_end_sec,
        config.input_world_timestamp_dir,
    )
    if end_frame < start_frame:
        raise ValueError(f"Invalid frame bounds for subject={subject} camera={camera}: {start_frame} > {end_frame}")
    return int(start_frame), int(end_frame), float(start_rel_time), float(end_rel_time)


def build_second_cut_plan(config: SyncConfig, subjects: list[str], cameras: list[str], residual_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    if residual_df.empty:
        print("No residual offsets available for second cut planning.")
        return pd.DataFrame(rows)

    first_cut_frames = load_first_cut_frame_map(config)
    second_pass_aligner = make_second_pass_aligner(config)

    for subject in subjects:
        drift_ref = choose_drift_reference(config, subject, cameras)
        if drift_ref is None:
            continue

        available: list[str] = []
        original_paths: dict[str, Path] = {}
        first_cut_entries: dict[str, tuple[int, int, float, float]] = {}
        source_durations: dict[str, float] = {}
        for camera in cameras:
            original_path = second_pass_aligner._find_video_path(subject, camera)
            if original_path is None:
                print(f"[skip] subject={subject} camera={camera}: missing original input video")
                continue
            try:
                first_cut_entries[camera] = first_cut_frame_entry(first_cut_frames, subject, camera)
            except KeyError as exc:
                print(f"[skip] {exc}")
                continue
            original_paths[camera] = original_path
            source_durations[camera] = original_video_duration_sec(second_pass_aligner, config, subject, camera)
            available.append(camera)

        if drift_ref not in available:
            print(f"[skip] subject={subject}: missing original video or first-cut frame map for drift reference {drift_ref}")
            continue

        corrected = corrected_camera_set(config, available, drift_ref)
        subject_residuals = residual_df[residual_df["subject"].astype(str) == str(subject)].copy()
        residual_by_camera = {str(row.camera): row for row in subject_residuals.itertuples(index=False)}

        front_offsets = {drift_ref: 0.0}
        back_offsets = {drift_ref: 0.0}
        drift_values = {drift_ref: 0.0}
        analysis_spans = {drift_ref: math.nan}
        audio_speed_factors = {drift_ref: 1.0}
        first_cut_durations = {}
        for camera, (_, _, first_start_sec, first_end_sec) in first_cut_entries.items():
            first_cut_duration = first_end_sec - first_start_sec
            if first_cut_duration <= 0:
                raise ValueError(f"Invalid first-cut duration for subject={subject} camera={camera}")
            first_cut_durations[camera] = first_cut_duration

        for camera in available:
            if camera == drift_ref:
                continue
            if camera not in residual_by_camera:
                print(f"[skip] subject={subject} camera={camera}: no residual offset row")
                continue
            row = subject_residuals[subject_residuals["camera"].astype(str) == str(camera)].iloc[0]
            front_offsets[camera] = float(row["front_offset_sec"])
            back_offsets[camera] = float(row["back_offset_sec"])
            drift_values[camera] = float(row["drift_sec"])
            analysis_spans[camera] = float(row["analysis_span_sec"])
            if camera in corrected:
                audio_speed_factors[camera] = speed_factor_from_drift(row, config)
            else:
                audio_speed_factors[camera] = 1.0

        planned_cameras = [
            camera
            for camera in available
            if camera in front_offsets and camera in back_offsets and camera in audio_speed_factors
        ]
        if len(planned_cameras) < 2:
            print(f"[skip] subject={subject}: fewer than two cameras available for second cut")
            continue

        common_start = max(front_offsets[camera] for camera in planned_cameras)
        common_end = min(first_cut_durations[camera] + back_offsets[camera] for camera in planned_cameras)
        if common_end <= common_start:
            raise ValueError(
                f"No positive endpoint-aligned overlap for subject={subject}: "
                f"common_start={common_start:.6f}s common_end={common_end:.6f}s"
            )

        start_trim = {camera: common_start - front_offsets[camera] for camera in planned_cameras}
        end_relative = {camera: common_end - back_offsets[camera] for camera in planned_cameras}

        for camera in planned_cameras:
            first_start_frame, first_end_frame, first_start_sec, first_end_sec = first_cut_entries[camera]
            start_rel_sec = float(start_trim[camera])
            end_rel_sec = float(end_relative[camera])
            first_cut_duration = first_cut_durations[camera]
            if start_rel_sec < -1e-6 or end_rel_sec > first_cut_duration + 1e-6:
                raise ValueError(
                    f"Endpoint-aligned cut is outside first-cut bounds for subject={subject} camera={camera}: "
                    f"start={start_rel_sec:.6f}s end={end_rel_sec:.6f}s duration={first_cut_duration:.6f}s"
                )
            start_rel_sec = max(0.0, start_rel_sec)
            end_rel_sec = min(first_cut_duration, end_rel_sec)
            if end_rel_sec <= start_rel_sec:
                raise ValueError(
                    f"Non-positive second-cut interval for subject={subject} camera={camera}: "
                    f"start={start_rel_sec:.6f}s end={end_rel_sec:.6f}s"
                )

            video_start_sec = first_start_sec + start_rel_sec
            video_end_sec = first_start_sec + end_rel_sec
            start_frame, end_frame, mapped_start_sec, mapped_end_sec = frame_bounds_from_original_times(
                second_pass_aligner,
                config,
                subject,
                camera,
                video_start_sec,
                video_end_sec,
            )
            audio_start_sec = video_start_sec
            audio_end_sec = video_end_sec
            audio_input_duration = audio_end_sec - audio_start_sec
            if audio_end_sec > source_durations[camera] + 1e-6:
                raise ValueError(
                    f"Second-cut audio end time exceeds original duration for subject={subject} camera={camera}: "
                    f"end={audio_end_sec:.6f}s duration={source_durations[camera]:.6f}s"
                )
            end_trim = max(0.0, first_cut_duration - end_rel_sec)
            source_interval_duration = video_end_sec - video_start_sec
            expected_audio_duration_after_tempo = audio_input_duration / audio_speed_factors[camera]
            rows.append(
                {
                    "subject": subject,
                    "camera": camera,
                    "drift_ref_cam": drift_ref,
                    "input_path": str(original_paths[camera]),
                    "output_path": str(corrected_output_path(config, subject, camera)),
                    "video_output_path": str(second_cut_video_path(config, subject, camera)),
                    "audio_output_path": str(second_cut_audio_path(config, subject, camera)),
                    "source_duration_sec": source_durations[camera],
                    "first_cut_start_frame": first_start_frame,
                    "first_cut_end_frame": first_end_frame,
                    "first_cut_start_sec": first_start_sec,
                    "first_cut_end_sec": first_end_sec,
                    "first_cut_duration_sec": first_cut_duration,
                    "front_offset_sec": front_offsets[camera],
                    "back_offset_sec": back_offsets[camera],
                    "drift_sec": drift_values[camera],
                    "analysis_span_sec": analysis_spans[camera],
                    "common_start_content_sec": common_start,
                    "common_end_content_sec": common_end,
                    "first_cut_relative_start_sec": start_rel_sec,
                    "first_cut_relative_end_sec": end_rel_sec,
                    "start_trim_sec": start_rel_sec,
                    "end_trim_sec": end_trim,
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "video_start_sec": video_start_sec,
                    "video_end_sec": video_end_sec,
                    "mapped_video_start_sec": mapped_start_sec,
                    "mapped_video_end_sec": mapped_end_sec,
                    "input_duration_sec": source_interval_duration,
                    "planned_source_duration_sec": source_interval_duration,
                    "audio_start_sec": audio_start_sec,
                    "audio_end_sec": audio_end_sec,
                    "audio_input_duration_sec": audio_input_duration,
                    "expected_audio_duration_after_tempo_sec": expected_audio_duration_after_tempo,
                    "audio_speed_factor": audio_speed_factors[camera],
                    "is_video_time_scaled": False,
                    "is_audio_time_scaled": abs(audio_speed_factors[camera] - 1.0) > 1e-6,
                }
            )

    plan = pd.DataFrame(rows)
    if not plan.empty:
        stage2_dir(config).mkdir(parents=True, exist_ok=True)
        plan.to_csv(stage2_dir(config) / "second_cut_compression_plan.csv", index=False)
        with open(stage2_dir(config) / "second_cut_compression_plan.pkl", "wb") as f:
            pickle.dump(plan.to_dict("records"), f)
    return plan


second_cut_plan = build_second_cut_plan(CONFIG, SUBJECTS, CAMERAS, residual_offsets)
display(second_cut_plan)


2026-06-17 21:00:53,820 | WARNING | second_cut_processing | World timestamp CSV missing for subject=expand0p1pctoffset1s camera=child. Using video fps fallback from /Users/renaissance/Desktop/2026_summer/ICDL/Evaluate-Video-Synchronizer/simulated_child_expansion_misalignment/videos/expand0p1pctoffset1s_child.mp4 (fps=29.958746 duration=121.200000).
2026-06-17 21:00:53,832 | INFO | second_cut_processing | subject=expand0p1pctoffset1s camera=child start_cut_sec=1.068135 -> start_frame=32 frame_time=1.068135 [fps fallback]
2026-06-17 21:00:53,833 | INFO | second_cut_processing | subject=expand0p1pctoffset1s camera=child end_cut_sec=0.066758 -> end_frame=3629 frame_time=121.133242 [fps fallback]
2026-06-17 21:00:53,909 | WARNING | second_cut_processing | World timestamp CSV missing for subject=expand0p1pctoffset1s camera=parent. Using video fps fallback from /Users/renaissance/Desktop/2026_summer/ICDL/Evaluate-Video-Synchronizer/simulated_child_expansion_misalignment/videos/expand0p1pctoff

,subject,camera,drift_ref_cam,input_path,output_path,video_output_path,audio_output_path,source_duration_sec,first_cut_start_frame,first_cut_end_frame,...,mapped_video_end_sec,input_duration_sec,planned_source_duration_sec,audio_start_sec,audio_end_sec,audio_input_duration_sec,expected_audio_duration_after_tempo_sec,audio_speed_factor,is_video_time_scaled,is_audio_time_scaled
0,expand0p1pctoffset1s,child,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,121.200000,32,3629,...,121.133242,120.065106,120.065106,1.068135,121.133242,120.065106,119.945161,1.001000,False,True
1,expand0p1pctoffset1s,parent,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,120.070033,0,3603,...,120.036718,119.955106,119.955106,0.067500,120.022606,119.955106,119.955106,1.000000,False,False
2,expand0p1pctoffset50ms,child,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,120.233333,3,3599,...,120.133195,120.033056,120.033056,0.100139,120.133195,120.033056,119.913143,1.001000,False,True
3,expand0p1pctoffset50ms,parent,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,120.070033,0,3603,...,119.970086,119.923056,119.923056,0.050000,119.973056,119.923056,119.923056,1.000000,False,False
4,expand1pctoffset1s,child,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,122.266667,55,3619,...,121.862591,120.010576,120.010576,1.852015,121.862591,120.010576,118.776915,1.010386,False,True
5,expand1pctoffset1s,parent,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,120.070033,0,3603,...,119.703560,118.868076,118.868076,0.842500,119.710576,118.868076,118.868076,1.000000,False,False
6,expand1pctoffset50ms,child,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,121.300000,26,3590,...,120.895891,120.020322,120.020322,0.875569,120.895891,120.020322,118.789234,1.010364,False,True
7,expand1pctoffset50ms,parent,parent,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,/Users/renaissance/Desktop/2026_summer/ICDL/Ev...,120.070033,0,3603,...,119.703560,118.880322,118.880322,0.815000,119.695322,118.880322,118.880322,1.000000,False,False


## Run Second Cut And Audio Tempo Correction

This cell creates the final corrected videos from the original input files. The video stream is cut by absolute frame index with `FrameAligner.cut_frames`, with no video speed correction. The audio stream is cut from the same original-time interval as the planned video cut, then corrected with the explicit planned `atempo=(span - drift) / span` factor before merging.


In [8]:
def atempo_filter(speed: float) -> str:
    if speed <= 0:
        raise ValueError(f"Audio tempo must be positive, got {speed}")
    factors: list[float] = []
    remaining = float(speed)
    while remaining > 2.0:
        factors.append(2.0)
        remaining /= 2.0
    while remaining < 0.5:
        factors.append(0.5)
        remaining /= 0.5
    factors.append(remaining)
    return ",".join(f"atempo={fmt_float(factor)}" for factor in factors)


def extract_tempo_adjusted_audio(
    input_path: Path,
    output_audio_path: Path,
    audio_start_sec: float,
    audio_input_duration_sec: float,
    tempo_factor: float,
    target_video_duration_sec: float,
    config: SyncConfig,
) -> float:
    if audio_input_duration_sec <= 0:
        raise ValueError(f"Audio input duration must be positive, got {audio_input_duration_sec}")
    if tempo_factor <= 0:
        raise ValueError(f"Audio tempo factor must be positive, got {tempo_factor}")
    if target_video_duration_sec <= 0:
        raise ValueError(f"Target video duration must be positive, got {target_video_duration_sec}")

    tempo = float(tempo_factor)
    af = (
        f"atrim=start={fmt_float(audio_start_sec)}:duration={fmt_float(audio_input_duration_sec)},"
        f"asetpts=PTS-STARTPTS,{atempo_filter(tempo)},"
        f"apad=whole_dur={fmt_float(target_video_duration_sec)}"
    )
    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        input_path,
        "-vn",
        "-af",
        af,
        "-t",
        fmt_float(target_video_duration_sec),
        "-acodec",
        "pcm_s16le",
        output_audio_path,
    ]
    run_command(cmd, context="extract and tempo-adjust second-cut audio")
    return tempo


def run_second_cut_and_audio_correction(plan: pd.DataFrame, config: SyncConfig) -> None:
    if plan.empty:
        print("No second-cut plan rows to run.")
        return

    stage2_dir(config).mkdir(parents=True, exist_ok=True)
    aligner = make_second_pass_aligner(config)
    processing_rows: list[dict[str, object]] = []
    for _, row in plan.iterrows():
        input_path = Path(row["input_path"])
        video_output_path = Path(row["video_output_path"])
        audio_output_path = Path(row["audio_output_path"])
        output_path = Path(row["output_path"])
        video_output_path.parent.mkdir(parents=True, exist_ok=True)
        audio_output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        start_frame = int(row["start_frame"])
        end_frame = int(row["end_frame"])
        audio_start_sec = float(row["audio_start_sec"])
        audio_input_duration_sec = float(row["audio_input_duration_sec"])
        audio_speed_factor = float(row["audio_speed_factor"])
        expected_audio_duration_after_tempo = float(row["expected_audio_duration_after_tempo_sec"])
        print(
            f"subject={row['subject']} camera={row['camera']} "
            f"frames={start_frame}-{end_frame} audio_start={audio_start_sec:.3f}s "
            f"audio_input_duration={audio_input_duration_sec:.3f}s atempo={audio_speed_factor:.6f} "
            f"expected_audio_after_tempo={expected_audio_duration_after_tempo:.3f}s"
        )

        aligner.cut_frames(input_path, video_output_path, start_frame, end_frame)
        video_duration_sec = media_duration_sec(video_output_path)
        actual_audio_tempo = extract_tempo_adjusted_audio(
            input_path=input_path,
            output_audio_path=audio_output_path,
            audio_start_sec=audio_start_sec,
            audio_input_duration_sec=audio_input_duration_sec,
            tempo_factor=audio_speed_factor,
            target_video_duration_sec=video_duration_sec,
            config=config,
        )
        audio_duration_sec = media_duration_sec(audio_output_path)
        aligner.merge_audio_video(video_output_path, audio_output_path, output_path)
        processing_rows.append(
            {
                "subject": row["subject"],
                "camera": row["camera"],
                "input_path": str(input_path),
                "video_output_path": str(video_output_path),
                "audio_output_path": str(audio_output_path),
                "output_path": str(output_path),
                "start_frame": start_frame,
                "end_frame": end_frame,
                "video_start_sec": float(row["video_start_sec"]),
                "video_end_sec": float(row["video_end_sec"]),
                "audio_start_sec": audio_start_sec,
                "audio_end_sec": float(row["audio_end_sec"]),
                "video_duration_sec": video_duration_sec,
                "audio_input_duration_sec": audio_input_duration_sec,
                "expected_audio_duration_after_tempo_sec": expected_audio_duration_after_tempo,
                "audio_duration_sec": audio_duration_sec,
                "planned_audio_speed_factor": audio_speed_factor,
                "actual_audio_tempo_factor": actual_audio_tempo,
            }
        )

    if processing_rows:
        pd.DataFrame(processing_rows).to_csv(stage2_dir(config) / "second_cut_processing_report.csv", index=False)


run_second_cut_and_audio_correction(second_cut_plan, CONFIG)


subject=expand0p1pctoffset1s camera=child frames=32-3629 audio_start=1.068s audio_input_duration=120.065s atempo=1.001000 expected_audio_after_tempo=119.945s
2026-06-17 21:04:55,017 | INFO | second_cut_processing | Saved cut video: /Users/renaissance/Desktop/2026_summer/ICDL/Evaluate-Video-Synchronizer/simulated_child_expansion_misalignment/notebook_eval_output/02_second_cut_stretch_corrected/expand0p1pctoffset1s_child_cut2_video.mp4
2026-06-17 21:04:56,900 | INFO | second_cut_processing | Saved merged video: /Users/renaissance/Desktop/2026_summer/ICDL/Evaluate-Video-Synchronizer/simulated_child_expansion_misalignment/notebook_eval_output/02_second_cut_stretch_corrected/expand0p1pctoffset1s_child_cut2_corrected.mp4
subject=expand0p1pctoffset1s camera=parent frames=2-3603 audio_start=0.068s audio_input_duration=119.955s atempo=1.000000 expected_audio_after_tempo=119.955s
2026-06-17 21:07:52,627 | INFO | second_cut_processing | Saved cut video: /Users/renaissance/Desktop/2026_summer/ICDL

## Verification After Audio Tempo Correction

Rerun the same front/middle/back residual measurement on the corrected videos. All three offsets should be close to zero and, most importantly, the back-minus-front drift should be much smaller after audio tempo correction.


In [9]:
def extract_corrected_audio_for_verification(config: SyncConfig, plan: pd.DataFrame) -> Path:
    verify_audio_dir = diagnostics_dir(config) / "corrected_audio"
    verify_audio_dir.mkdir(parents=True, exist_ok=True)
    for _, row in plan.iterrows():
        video_path = Path(row["output_path"])
        if not video_path.exists():
            continue
        wav_path = verify_audio_dir / f"{row['subject']}_{row['camera']}_cut.wav"
        cmd = [
            "ffmpeg",
            "-y",
            "-i",
            video_path,
            "-vn",
            "-acodec",
            "pcm_s16le",
            "-ac",
            "1",
        ]
        if config.audio_sr is not None:
            cmd.extend(["-ar", str(config.audio_sr)])
        cmd.append(wav_path)
        run_command(cmd, context="extract corrected audio for verification")
    return verify_audio_dir


def measure_corrected_outputs(config: SyncConfig, plan: pd.DataFrame) -> pd.DataFrame:
    if plan.empty:
        return pd.DataFrame()

    verify_audio_dir = extract_corrected_audio_for_verification(config, plan)
    def patched_first_cut_audio_path(subject: str, camera: str) -> Path:
        return verify_audio_dir / f"{subject}_{camera}_cut.wav"

    rows: list[dict[str, object]] = []
    aligner = make_analysis_aligner(config)
    subjects = sorted({str(subject) for subject in plan["subject"].unique()})
    cameras = sorted({str(camera) for camera in plan["camera"].unique()})

    for subject in subjects:
        drift_ref = choose_drift_reference(config, subject, cameras)
        if drift_ref is None:
            subject_rows = plan[plan["subject"].astype(str) == subject]
            if subject_rows.empty:
                continue
            drift_ref = str(subject_rows.iloc[0]["drift_ref_cam"])

        ref_path = patched_first_cut_audio_path(subject, drift_ref)
        if not ref_path.exists():
            continue
        ref_audio, sr_ref = load_audio_mono(ref_path, config.audio_sr)
        ref_duration = len(ref_audio) / sr_ref

        for camera in cameras:
            if camera == drift_ref:
                continue
            cmp_path = patched_first_cut_audio_path(subject, camera)
            if not cmp_path.exists():
                continue
            cmp_audio, sr_cmp = load_audio_mono(cmp_path, config.audio_sr)
            if sr_cmp != sr_ref:
                raise ValueError(f"Sample-rate mismatch for corrected subject={subject} camera={camera}")
            common_duration = min(ref_duration, len(cmp_audio) / sr_cmp)
            if common_duration < config.residual_window_sec * 3:
                continue

            front_start = 0.0
            middle_start = max(0.0, (common_duration - config.residual_window_sec) / 2.0)
            back_start = max(0.0, common_duration - config.residual_window_sec)
            front_offset, _ = estimate_offset_sec(
                aligner,
                audio_window(ref_audio, sr_ref, front_start, config.residual_window_sec),
                audio_window(cmp_audio, sr_cmp, front_start, config.residual_window_sec),
                sr_ref,
                config,
            )
            middle_offset, _ = estimate_offset_sec(
                aligner,
                audio_window(ref_audio, sr_ref, middle_start, config.residual_window_sec),
                audio_window(cmp_audio, sr_cmp, middle_start, config.residual_window_sec),
                sr_ref,
                config,
            )
            back_offset, _ = estimate_offset_sec(
                aligner,
                audio_window(ref_audio, sr_ref, back_start, config.residual_window_sec),
                audio_window(cmp_audio, sr_cmp, back_start, config.residual_window_sec),
                sr_ref,
                config,
            )
            rows.append(
                {
                    "subject": subject,
                    "drift_ref_cam": drift_ref,
                    "camera": camera,
                    "front_offset_sec": front_offset,
                    "middle_offset_sec": middle_offset,
                    "back_offset_sec": back_offset,
                    "drift_sec": back_offset - front_offset,
                }
            )

    corrected_df = pd.DataFrame(rows)
    corrected_df.to_csv(diagnostics_dir(config) / "corrected_front_back_offsets.csv", index=False)
    return corrected_df


corrected_offsets = measure_corrected_outputs(CONFIG, second_cut_plan)
display(corrected_offsets)


,subject,drift_ref_cam,camera,front_offset_sec,middle_offset_sec,back_offset_sec,drift_sec
0,expand0p1pctoffset1s,parent,child,0.0050,0.005,0.0000,-0.0050
1,expand0p1pctoffset50ms,parent,child,0.0025,0.010,0.0025,0.0000
2,expand1pctoffset1s,parent,child,0.0050,0.030,0.0475,0.0425
3,expand1pctoffset50ms,parent,child,0.0075,0.030,0.0475,0.0400
